In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Merged Adapter Evaluation: MetaMathQA (GSM8K) + CodeAlpaca (HumanEval) + Dolly-15k (Perplexity)

In [ ]:
import torch
import gc
import re
import math
import multiprocessing
import contextlib
import io
from datasets import load_dataset
from tqdm import tqdm
from unsloth import FastLanguageModel

# ─────────────────────────────────────────────
# SHARED: chat-template prompt builder
# CRITICAL FIX: enable_thinking=False — Qwen3's <think> block otherwise
# eats the entire max_new_tokens budget before any answer is produced.
# ─────────────────────────────────────────────

def build_chat_prompt(tokenizer, user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def prep_tokenizer_for_generation(tokenizer):
    """Left-padding is required for correct batched causal-LM generation."""
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


# ─────────────────────────────────────────────
# GSM8K EVALUATION (exact match)
# ─────────────────────────────────────────────

def extract_gsm8k_answer(text: str) -> str:
    """Extract final answer — tries '#### X' first, then last number."""
    hash_match = re.findall(r"####\s*(-?[\d,]+\.?\d*)", text)
    if hash_match:
        return hash_match[-1].replace(",", "").strip()
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].replace(",", "").strip()
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return text.strip()


def format_gsm8k_prompt(question: str) -> str:
    return (
        f"Solve the following math problem. Show your reasoning and put "
        f"your final numeric answer after '#### '.\n\n"
        f"Question: {question}"
    )


def evaluate_gsm8k(
    model,
    tokenizer,
    model_name: str = "model",
    num_samples: int = 200,
    batch_size: int = 4,
    max_new_tokens: int = 320,
    device: str = "cuda",
) -> dict:
    """Evaluate on GSM8K (test split) using exact match on final answer."""
    print(f"\n{'─'*60}")
    print(f"[GSM8K] Evaluating: {model_name}")
    print(f"{'─'*60}")

    model.eval()
    prep_tokenizer_for_generation(tokenizer)

    dataset = load_dataset("gsm8k", "main", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))

    preds, labels = [], []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [gsm8k]"):
        batch = dataset[i : i + batch_size]

        questions    = batch["question"]
        true_answers = [extract_gsm8k_answer(a) for a in batch["answer"]]
        prompts      = [build_chat_prompt(tokenizer, format_gsm8k_prompt(q)) for q in questions]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
            add_special_tokens=False,  # chat template already adds special tokens
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
            )

        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated   = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred_answer = extract_gsm8k_answer(generated)
            preds.append(pred_answer)
            labels.append(true_answers[j])

    per_sample_exact = [int(p.strip() == l.strip()) for p, l in zip(preds, labels)]
    exact_match      = round(sum(per_sample_exact) / len(per_sample_exact), 4)

    result = {
        "repo_id"          : model_name,
        "exact_match"      : exact_match,
        "num_samples"      : len(per_sample_exact),
        "per_sample_exact" : per_sample_exact,
        "sample_preds"     : list(zip(labels[:5], preds[:5])),
    }

    print(f"  Exact Match: {exact_match:.4f}")
    for true, pred in result["sample_preds"]:
        print(f"    {true:<15} → {pred}")

    return result


# ─────────────────────────────────────────────
# HUMANEVAL EVALUATION (pass@1)
# ─────────────────────────────────────────────

def extract_code(generated: str, problem_prompt: str, entry_point: str) -> str:
    """Pull a runnable function body/definition out of raw model output."""
    text = generated.strip()
    fence = re.search(r"```(?:python)?\s*\n?(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    if f"def {entry_point}" in text:
        return text
    # Model only continued the body — stitch it onto the given signature.
    return problem_prompt + "\n" + text


def _unsafe_execute(program: str, result_list, timeout: int):
    import signal

    def handler(signum, frame):
        raise TimeoutError("execution timed out")

    try:
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(timeout)
        exec_globals = {}
        with contextlib.redirect_stdout(io.StringIO()):
            exec(program, exec_globals)
        signal.alarm(0)
        result_list.append("passed")
    except Exception as e:
        result_list.append(f"failed: {type(e).__name__}: {e}")


def check_correctness(problem: dict, completion_code: str, timeout: int = 5) -> bool:
    """Run generated code + the HumanEval test harness in an isolated process."""
    program = (
        completion_code
        + "\n"
        + problem["test"]
        + f"\ncheck({problem['entry_point']})\n"
    )
    manager = multiprocessing.Manager()
    result_list = manager.list()
    p = multiprocessing.Process(target=_unsafe_execute, args=(program, result_list, timeout))
    p.start()
    p.join(timeout=timeout + 1)
    if p.is_alive():
        p.kill()
        p.join()
    if not result_list:
        result_list.append("failed: timeout")
    return result_list[0] == "passed"


def format_humaneval_prompt(problem_prompt: str) -> str:
    return (
        "Complete the following Python function. Return ONLY the complete "
        "function code (including the signature), with no explanations and "
        "no markdown formatting.\n\n"
        f"{problem_prompt}"
    )


def evaluate_humaneval(
    model,
    tokenizer,
    model_name: str = "model",
    num_samples: int = 164,   # full HumanEval set
    batch_size: int = 4,
    max_new_tokens: int = 384,
    device: str = "cuda",
) -> dict:
    """Evaluate on OpenAI HumanEval using pass@1 (greedy, single sample)."""
    print(f"\n{'─'*60}")
    print(f"[HumanEval] Evaluating: {model_name}")
    print(f"{'─'*60}")

    model.eval()
    prep_tokenizer_for_generation(tokenizer)

    dataset = load_dataset("openai/openai_humaneval", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))

    per_sample_pass = []
    sample_results  = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [humaneval]"):
        batch = dataset[i : i + batch_size]

        problem_prompts = batch["prompt"]
        entry_points    = batch["entry_point"]
        prompts         = [build_chat_prompt(tokenizer, format_humaneval_prompt(p)) for p in problem_prompts]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=768,
            add_special_tokens=False,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
            )

        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            code      = extract_code(generated, problem_prompts[j], entry_points[j])

            problem = {
                "prompt"      : problem_prompts[j],
                "test"        : batch["test"][j],
                "entry_point" : entry_points[j],
            }
            passed = check_correctness(problem, code, timeout=5)
            per_sample_pass.append(int(passed))
            sample_results.append((batch["task_id"][j], passed))

    pass_at_1 = round(sum(per_sample_pass) / len(per_sample_pass), 4)

    result = {
        "repo_id"        : model_name,
        "pass_at_1"      : pass_at_1,
        "num_samples"    : len(per_sample_pass),
        "per_sample_pass": per_sample_pass,
        "sample_results" : sample_results[:5],
    }

    print(f"  pass@1: {pass_at_1:.4f}")
    for task_id, passed in result["sample_results"]:
        print(f"    {task_id:<15} → {'PASS' if passed else 'FAIL'}")

    return result


# ─────────────────────────────────────────────
# DOLLY-15K EVALUATION (perplexity, response tokens only)
# Looped one sample at a time (no padding) — avoids the ValueError from
# return_tensors="pt" on variable-length batches seen in earlier harnesses.
# ─────────────────────────────────────────────

def format_dolly_prompt(instruction: str, context: str) -> str:
    if context:
        return f"Instruction: {instruction}\nContext: {context}\nResponse:"
    return f"Instruction: {instruction}\nResponse:"


def evaluate_dolly_perplexity(
    model,
    tokenizer,
    model_name: str = "model",
    num_samples: int = 200,
    max_length: int = 512,
    device: str = "cuda",
) -> dict:
    """Evaluate on Dolly-15k using perplexity over response tokens only
    (prompt tokens masked out with label = -100)."""
    print(f"\n{'─'*60}")
    print(f"[Dolly-PPL] Evaluating: {model_name}")
    print(f"{'─'*60}")

    model.eval()

    dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))

    per_sample_nll = []
    per_sample_ppl = []

    for ex in tqdm(dataset, desc=f"{model_name} [dolly-ppl]"):
        prompt = format_dolly_prompt(ex["instruction"], ex.get("context", ""))
        response = ex["response"]
        if not response.strip():
            continue
        full_text = prompt + " " + response

        prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        full_ids   = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)

        if full_ids.shape[1] <= prompt_ids.shape[1]:
            continue  # response truncated away entirely, skip

        labels = full_ids.clone()
        labels[:, : prompt_ids.shape[1]] = -100

        with torch.no_grad():
            out = model(full_ids, labels=labels)

        nll = out.loss.item()
        per_sample_nll.append(nll)
        per_sample_ppl.append(math.exp(nll))

    result = {
        "repo_id"         : model_name,
        "perplexity"      : round(sum(per_sample_ppl) / len(per_sample_ppl), 4),
        "mean_nll"        : round(sum(per_sample_nll) / len(per_sample_nll), 4),
        "num_samples"     : len(per_sample_ppl),
        "per_sample_nll"  : per_sample_nll,   # use THIS for bootstrap (lower = better, additive)
        "per_sample_ppl"  : per_sample_ppl,   # reporting only — not additive across samples
    }

    print(f"  Perplexity: {result['perplexity']:.4f}  (mean NLL: {result['mean_nll']:.4f})")

    return result


# ─────────────────────────────────────────────
# EVALUATE ALL MERGED MODELS
# ─────────────────────────────────────────────

def evaluate_all_qwen_models(
    repos: list,
    num_samples_gsm8k: int = 200,
    num_samples_humaneval: int = 164,
    num_samples_dolly: int = 200,
    batch_size: int = 4,
    device: str = "cuda",
) -> dict:
    all_results = {}

    for repo in repos:
        print(f"\n{'═'*60}")
        print(f"Model: {repo}")
        print(f"{'═'*60}")

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name     = repo,
            max_seq_length = 1024,
            load_in_4bit   = True,
            dtype          = torch.float16,
        )
        FastLanguageModel.for_inference(model)

        gsm8k_result     = evaluate_gsm8k(model, tokenizer, model_name=repo,
                                           num_samples=num_samples_gsm8k, batch_size=batch_size)
        humaneval_result = evaluate_humaneval(model, tokenizer, model_name=repo,
                                               num_samples=num_samples_humaneval, batch_size=batch_size)
        dolly_result      = evaluate_dolly_perplexity(model, tokenizer, model_name=repo,
                                                        num_samples=num_samples_dolly)

        all_results[repo] = {
            "gsm8k"    : gsm8k_result,
            "humaneval": humaneval_result,
            "dolly_ppl": dolly_result,
        }

        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    # Summary table
    print(f"\n{'═'*70}")
    print(f"{'MODEL':<35} {'GSM8K':>8} {'pass@1':>8} {'PPL':>8}")
    print(f"{'─'*70}")
    for repo, r in all_results.items():
        name = repo.split("/")[-1]
        print(
            f"{name:<35} "
            f"{r['gsm8k']['exact_match']:>8.4f} "
            f"{r['humaneval']['pass_at_1']:>8.4f} "
            f"{r['dolly_ppl']['perplexity']:>8.4f}"
        )
    print(f"{'═'*70}")

    return all_results


# ── Run evaluation ──
repos = [
    "Srishtik/Qwen3-0.6B-linear-3-adapters-merged-2",
    "Srishtik/Qwen3-0.6B-svd-3-adapters-merged-2",
    "Srishtik/Qwen3-0.6B-ties-3-adapters-merged-2",
    "Srishtik/Qwen3-0.6B-dare-3-adapters-merged-2",
    "Srishtik/Qwen3-0.6B-slerp-3-adapters-merged-2",
]

all_results = evaluate_all_qwen_models(
    repos                 = repos,
    num_samples_gsm8k     = 200,
    num_samples_humaneval = 164,
    num_samples_dolly     = 200,
    batch_size            = 4,
)


## Evaluate Reference / Baseline Models

Run the cells below **after** `evaluate_all_qwen_models()` has completed. Results are added directly to `all_results` so the bootstrap and heatmap cells include them automatically.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# evaluate_initialized_model_all
# Use this for any already-loaded model (full model, individual specialist
# adapters, etc.). Runs GSM8K (EM), HumanEval (pass@1), Dolly-15k (perplexity).
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_initialized_model_all(
    model,
    tokenizer,
    model_name: str = "model",
    num_samples_gsm8k: int = 200,
    num_samples_humaneval: int = 164,
    num_samples_dolly: int = 200,
    batch_size: int = 4,
    device: str = "cuda",
) -> dict:
    gsm8k_result = evaluate_gsm8k(
        model, tokenizer,
        model_name  = model_name,
        num_samples = num_samples_gsm8k,
        batch_size  = batch_size,
        device      = device,
    )
    humaneval_result = evaluate_humaneval(
        model, tokenizer,
        model_name  = model_name,
        num_samples = num_samples_humaneval,
        batch_size  = batch_size,
        device      = device,
    )
    dolly_result = evaluate_dolly_perplexity(
        model, tokenizer,
        model_name  = model_name,
        num_samples = num_samples_dolly,
    )

    print(f"\n  ── {model_name} summary ──")
    print(f"  GSM8K Exact Match : {gsm8k_result['exact_match']:.4f}")
    print(f"  HumanEval pass@1  : {humaneval_result['pass_at_1']:.4f}")
    print(f"  Dolly Perplexity  : {dolly_result['perplexity']:.4f}")

    return {
        "gsm8k"    : gsm8k_result,
        "humaneval": humaneval_result,
        "dolly_ppl": dolly_result,
    }


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Evaluate reference / baseline models and add to all_results
#
# Run AFTER evaluate_all_qwen_models() has populated all_results.
#
# Three baselines are evaluated:
#   1. codealpaca_adapter — specialist adapter trained only on CodeAlpaca
#   2. metamath_adapter   — specialist adapter trained only on MetaMathQA
#   3. dolly_adapter      — specialist adapter trained only on Dolly-15k
#
# Add or remove entries below as needed.
# ─────────────────────────────────────────────────────────────────────────────

from unsloth import FastLanguageModel
import torch, gc

def load_and_evaluate(repo_id, model_name, num_samples_gsm8k=200, num_samples_humaneval=164,
                       num_samples_dolly=200, batch_size=4):
    """Load a HuggingFace repo and evaluate it on all three tasks."""
    print(f"\n{'═'*60}")
    print(f"Loading: {repo_id}")
    print(f"{'═'*60}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = repo_id,
        max_seq_length = 1024,
        load_in_4bit   = True,
        dtype          = torch.float16,
    )
    FastLanguageModel.for_inference(model)

    result = evaluate_initialized_model_all(
        model       = model,
        tokenizer   = tokenizer,
        model_name  = model_name,
        num_samples_gsm8k     = num_samples_gsm8k,
        num_samples_humaneval = num_samples_humaneval,
        num_samples_dolly     = num_samples_dolly,
        batch_size  = batch_size,
    )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return result


# ── CodeAlpaca specialist adapter ─────────────────────────────────────────────
codealpaca_result = load_and_evaluate(
    repo_id    = "Srishtik/qwen3-trained-on-code-alpaca-18k",
    model_name = "codealpaca_adapter",
)
all_results["codealpaca_adapter"] = codealpaca_result

# ── MetaMathQA specialist adapter ─────────────────────────────────────────────
metamath_result = load_and_evaluate(
    repo_id    = "Srishtik/qwen3-trained-on-metamath-15k",
    model_name = "metamath_adapter",
)
all_results["metamath_adapter"] = metamath_result

# ── Dolly-15k specialist adapter ──────────────────────────────────────────────
dolly_result = load_and_evaluate(
    repo_id    = "Srishtik/qwen3-trained-on-dolly-15k",
    model_name = "dolly_adapter",
)
all_results["dolly_adapter"] = dolly_result

# ── Full model (all three tasks jointly, if you have one) ────────────────────
# Uncomment and update repo name if you have a jointly-trained reference:
# full_model_result = load_and_evaluate(
#     repo_id    = "Srishtik/Qwen3-0.6B-full-3task-model",
#     model_name = "full_model",
# )
# all_results["full_model"] = full_model_result

print(f"\nall_results now contains {len(all_results)} models:")
for k in all_results:
    em  = all_results[k]['gsm8k']['exact_match']
    p1  = all_results[k]['humaneval']['pass_at_1']
    ppl = all_results[k]['dolly_ppl']['perplexity']
    print(f"  {k:<35} GSM8K={em:.4f}  pass@1={p1:.4f}  PPL={ppl:.4f}")


## Bootstrap Statistical Significance Tests

Pairwise bootstrap t-tests (n=10,000 resamples) across all models for GSM8K Exact Match, HumanEval pass@1, and Dolly-15k Mean NLL.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BOOTSTRAP T-TEST: Pairwise significance across all models + baselines
# Metrics: Exact Match (GSM8K) | pass@1 (HumanEval) | mean NLL (Dolly-15k)
# NOTE: for NLL, LOWER is better — use higher_is_better=False so the
# "X > Y" significance summary reports the correct winner.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
from itertools import combinations

def bootstrap_ttest(scores_a, scores_b, n_bootstrap=10000, seed=42):
    rng  = np.random.default_rng(seed)
    a    = np.array(scores_a, dtype=float)
    b    = np.array(scores_b, dtype=float)
    n    = len(a)
    assert n == len(b), "Score arrays must be same length"

    observed_diff = a.mean() - b.mean()
    boot_diffs    = [a[rng.integers(0, n, n)].mean() - b[rng.integers(0, n, n)].mean()
                     for _ in range(n_bootstrap)]
    boot_diffs    = np.array(boot_diffs)
    p_value       = np.mean(np.abs(boot_diffs) >= np.abs(observed_diff))
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

    return {
        "diff"   : round(float(observed_diff), 4),
        "p_value": round(float(p_value), 4),
        "ci_low" : round(float(ci_low),  4),
        "ci_high": round(float(ci_high), 4),
        "sig"    : "***" if p_value < 0.001 else ("**" if p_value < 0.01 else ("*" if p_value < 0.05 else "ns")),
    }


def run_bootstrap_for_metric(all_results, metric, get_scores_fn, label, n_bootstrap=10000,
                              higher_is_better=True):
    baseline_keys = {"codealpaca_adapter", "metamath_adapter", "dolly_adapter", "full_model"}
    merged_keys   = [k for k in all_results if k not in baseline_keys]
    ref_keys      = [k for k in all_results if k in baseline_keys]
    all_keys      = merged_keys + ref_keys

    def short(k):
        return (k.replace("Srishtik/", "")
                  .replace("Qwen3-0.6B-", "")
                  .replace("-3-adapters-merged-2", ""))

    scores = {}
    for k in all_keys:
        try:
            scores[k] = get_scores_fn(all_results[k])
        except (KeyError, TypeError):
            print(f"  [skip] {k} has no per-sample scores for {metric}")

    def print_section(title, keys_a, keys_b=None):
        pairs = list(combinations(keys_a, 2)) if keys_b is None else [(a, b) for a in keys_a for b in keys_b]
        pairs = [(a, b) for a, b in pairs if a in scores and b in scores]
        if not pairs:
            return []
        print(f"\n  {'─'*98}")
        print(f"  {title}")
        print(f"  {'─'*98}")
        print(f"  {'Model A':<35} {'Model B':<35} {'Diff':>7} {'p-val':>7} {'95% CI':>22} {'Sig':>5}")
        print(f"  {'─'*98}")
        table = []
        for na, nb_ in pairs:
            r      = bootstrap_ttest(scores[na], scores[nb_], n_bootstrap=n_bootstrap)
            ci_str = f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]"
            print(f"  {short(na):<35} {short(nb_):<35} {r['diff']:>+7.4f} {r['p_value']:>7.4f} {ci_str:>22} {r['sig']:>5}")
            table.append({"metric": metric, "model_a": short(na), "model_b": short(nb_), **r})
        return table

    print(f"\n{'═'*100}")
    print(f"BOOTSTRAP T-TEST — {label}  (n_bootstrap={n_bootstrap:,})")
    print(f"Significance: * p<0.05   ** p<0.01   *** p<0.001   ns = not significant")
    if not higher_is_better:
        print(f"NOTE: lower is better for this metric — diff > 0 means Model A is WORSE.")
    print(f"{'═'*100}")

    table = []
    table += print_section("MERGED MODELS — pairwise",               merged_keys)
    if ref_keys:
        table += print_section("MERGED MODELS vs REFERENCE ADAPTERS", merged_keys, ref_keys)
        if len(ref_keys) > 1:
            table += print_section("REFERENCE ADAPTERS — pairwise",   ref_keys)

    sig    = [r for r in table if r["sig"] != "ns"]
    nonsig = [r for r in table if r["sig"] == "ns"]
    print(f"\n  Total: {len(table)} pairs | Significant: {len(sig)} | Non-significant: {len(nonsig)}")
    if sig:
        for r in sig:
            a_better = (r["diff"] > 0) == higher_is_better
            d = f"{r['model_a']} > {r['model_b']}" if a_better else f"{r['model_b']} > {r['model_a']}"
            print(f"    {d}  (diff={r['diff']:+.4f}, p={r['p_value']:.4f}, {r['sig']})")
    if not sig:
        print("    No significant differences on this metric.")

    return table


# ── Run for all three metrics ───────────────────────────────────────────────────
all_bootstrap_results = []

all_bootstrap_results += run_bootstrap_for_metric(
    all_results, "exact_match",
    get_scores_fn = lambda r: r["gsm8k"]["per_sample_exact"],
    label         = "EXACT MATCH (GSM8K)",
    higher_is_better = True,
)
all_bootstrap_results += run_bootstrap_for_metric(
    all_results, "pass_at_1",
    get_scores_fn = lambda r: r["humaneval"]["per_sample_pass"],
    label         = "pass@1 (HumanEval)",
    higher_is_better = True,
)
all_bootstrap_results += run_bootstrap_for_metric(
    all_results, "mean_nll",
    get_scores_fn = lambda r: r["dolly_ppl"]["per_sample_nll"],
    label         = "Mean NLL (Dolly-15k)",
    higher_is_better = False,
)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HEATMAP: p-value and score-difference matrices for all 3 metrics
# Merged models and reference adapters separated by a divider line
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

baseline_keys = {"codealpaca_adapter", "metamath_adapter", "dolly_adapter", "full_model"}
merged_keys   = [k for k in all_results if k not in baseline_keys]
ref_keys      = [k for k in all_results if k in baseline_keys]
all_keys      = merged_keys + ref_keys
n_merged      = len(merged_keys)

def short(k):
    return (k.replace("Srishtik/", "")
              .replace("Qwen3-0.6B-", "")
              .replace("-3-adapters-merged-2", ""))

short_names = [short(k) for k in all_keys]
n           = len(all_keys)

metrics = {
    "GSM8K Exact Match" : lambda r: r["gsm8k"]["per_sample_exact"],
    "HumanEval pass@1"  : lambda r: r["humaneval"]["per_sample_pass"],
    "Dolly Mean NLL"    : lambda r: r["dolly_ppl"]["per_sample_nll"],  # lower is better
}

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle("Bootstrap T-Test Results (n=10,000 resamples)\nTop: p-values | Bottom: score differences (row−col)", fontsize=13, fontweight="bold")

for col, (metric_name, get_fn) in enumerate(metrics.items()):
    scores = {}
    for k in all_keys:
        try:
            scores[k] = get_fn(all_results[k])
        except (KeyError, TypeError):
            scores[k] = None

    p_mat    = np.ones((n, n))
    diff_mat = np.zeros((n, n))

    for i, na in enumerate(all_keys):
        for j, nb_ in enumerate(all_keys):
            if i != j and scores[na] is not None and scores[nb_] is not None:
                r = bootstrap_ttest(scores[na], scores[nb_], n_bootstrap=10000)
                p_mat[i, j]    = r["p_value"]
                diff_mat[i, j] = r["diff"]

    for row_idx, (matrix, ax, cmap, vmin, vmax, cb_label, fmt) in enumerate([
        (p_mat,    axes[0, col], "RdYlGn_r", 0,    0.1,  "p-value", ".3f"),
        (diff_mat, axes[1, col], "RdBu",     None, None, "Diff",    "+.3f"),
    ]):
        if vmin is None:
            lim  = max(0.05, float(np.abs(matrix).max()))
            vmin, vmax = -lim, lim

        im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks(range(n)); ax.set_yticks(range(n))
        ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=7)
        ax.set_yticklabels(short_names, fontsize=7)
        title_suffix = "p-values" if row_idx == 0 else "Score diff (row−col)"
        ax.set_title(f"{metric_name}\n{title_suffix}", fontsize=9)

        for i in range(n):
            for j in range(n):
                val   = matrix[i, j]
                txt   = "—" if i == j else f"{val:{fmt}}"
                color = "white" if (row_idx == 0 and val < 0.05) else "black"
                ax.text(j, i, txt, ha="center", va="center", fontsize=6.5, color=color)

        # Divider between merged and reference models
        if n_merged < n:
            ax.axhline(n_merged - 0.5, color="black", linewidth=2)
            ax.axvline(n_merged - 0.5, color="black", linewidth=2)

        plt.colorbar(im, ax=ax, fraction=0.046, label=cb_label)

# Legend
p1 = mpatches.Patch(color="white",    edgecolor="black", label="Above divider: merged models")
p2 = mpatches.Patch(color="lightgrey",edgecolor="black", label="Below divider: reference adapters")
fig.legend(handles=[p1, p2], loc="lower center", ncol=2, fontsize=9, frameon=True)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig("bootstrap_heatmaps_with_baselines.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: bootstrap_heatmaps_with_baselines.png")
